# AE model

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# =========================
# Masking Function
# =========================
def apply_mask(x, mask_ratio=0.2):
    """
    x: (batch, seq_len, num_features)
    """
    x_masked = x.clone()

    # ---- feature-level masking ----
    mask = torch.rand_like(x) < mask_ratio
    x_masked[mask] = 0.0

    # ---- timestep-level masking (کل یک timestep) ----
    if np.random.rand() < 0.3:
        t = np.random.randint(0, x.shape[1])
        x_masked[:, t, :] = 0.0

        # مهم: این timestep باید در loss هم لحاظ شود
        mask[:, t, :] = True

    #print(x.shape)    
    #print(x_masked.shape)    
    return x_masked, mask


# =========================
# Attention Pooling
# =========================
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)

    def forward(self, x):
        # x: (B, T, D)
        weights = torch.softmax(self.attn(x), dim=1)  # (B, T, 1)
        pooled = (x * weights).sum(dim=1)             # (B, D)
        return pooled


# =========================================================
# Temporal Conv Block
# =========================================================

class TemporalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = nn.ReLU()

    def forward(self, x):
        # x: (B, T, F) → (B, F, T)
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        # back to (B, T, F)
        return x.permute(0, 2, 1)


# =========================================================
# Gated Attention Pooling
# =========================================================

class GatedAttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )

        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )

    def forward(self, x):

        # x: (B, T, D)

        gate = self.gate(x)

        x = x * gate

        scores = self.attn(x)

        weights = torch.softmax(scores, dim=1)

        pooled = torch.sum(weights * x, dim=1)

        return pooled


# =========================================================
# Improved Temporal Encoder
# =========================================================

class ImprovedTemporalAutoencoder(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=AE_Hidden_Dim,
        embed_dim=embed_Ftr_dim,
        num_heads=4,
        dropout=0.2
    ):

        super().__init__()

        # =================================================
        # TCN Branch
        # =================================================

        self.tcn1 = TemporalConvBlock(
            input_dim,
            hidden_dim,
            #dilation=1
        )

        self.tcn2 = TemporalConvBlock(
            hidden_dim,
            hidden_dim,
            #dilation=2
        )

        # =================================================
        # BiGRU Branch
        # =================================================

        self.bigru = nn.GRU(
            input_dim,
            hidden_dim // 2,
            batch_first=True,
            bidirectional=True
        )

        # =================================================
        # Fusion
        # =================================================

        self.fusion = nn.Linear(
            hidden_dim * 2,
            hidden_dim
        )

        # =================================================
        # Lightweight Self Attention
        # =================================================

        self.mha = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            batch_first=True,
            dropout=dropout
        )

        # =================================================
        # LayerNorm
        # =================================================

        self.norm1 = nn.LayerNorm(hidden_dim)

        # =================================================
        # Pooling
        # =================================================

        #self.pool = GatedAttentionPooling(hidden_dim)
        self.pool = AttentionPooling(hidden_dim)

        # =================================================
        # Embedding
        # =================================================

        self.embedding = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        # =================================================
        # Better Decoder
        # =================================================

        self.decoder_gru = nn.GRU(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.decoder_out = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

        self.dropout = nn.Dropout(dropout)

    # =====================================================
    # Forward
    # =====================================================

    def forward(self, x):

        # x: (B, T, F)

        B, T, Fdim = x.shape

        # -------------------------------------------------
        # TCN branch
        # -------------------------------------------------

        tcn = self.tcn1(x)
        tcn = self.dropout(tcn)

        tcn = self.tcn2(tcn)
        tcn = self.dropout(tcn)

        # -------------------------------------------------
        # GRU branch
        # -------------------------------------------------

        gru, _ = self.bigru(x)

        gru = self.dropout(gru)

        # -------------------------------------------------
        # Fusion
        # -------------------------------------------------

        fused = torch.cat([tcn, gru], dim=-1)
        #print(tcn.shape)
        #print(gru.shape)

        fused = self.fusion(fused)

        # -------------------------------------------------
        # Self Attention
        # -------------------------------------------------

        attn_out, _ = self.mha(
            fused,
            fused,
            fused
        )

        fused = self.norm1(fused + attn_out)

        # -------------------------------------------------
        # Pooling
        # -------------------------------------------------

        pooled = self.pool(fused)

        # -------------------------------------------------
        # Embedding
        # -------------------------------------------------

        emb = self.embedding(pooled)

        # -------------------------------------------------
        # Decoder
        # -------------------------------------------------

        dec_in = emb.unsqueeze(1).repeat(1, T, 1)

        dec_out, _ = self.decoder_gru(dec_in)

        recon = self.decoder_out(dec_out)

        return emb, recon
    
    
    
# =========================
# Loss Function (masked only)
# =========================
def masked_mse_loss(pred, target, mask):
    loss = (pred - target) ** 2
    loss = loss[mask]  # only masked positions
    return loss.mean()


def evaluate(model, dataloader, device=DEV):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            x, _ = batch
            x = x.to(device)

            x_masked, mask = apply_mask(x, mask_ratio=0.2)

            _, recon = model(x_masked)

            loss = masked_mse_loss(recon, x, mask)
            total_loss += loss.item()

    return total_loss / len(dataloader)    